# Estimating Effects

Este notebook estima el efecto causal de las cámaras de velocidad sobre los incidentes viales utilizando el método de diferencia en diferencias (difference-in-differences). Calcula los efectos tanto para el número total de accidentes como para las tasas, y para diferentes niveles de severidad (total, min, pic, fcs). Incluye modelos con y sin efectos fijos.

**Inputs necesarios:**
- `outcome-variables.parquet`: Variables de resultado (accidentes) por grid y tiempo
- `matched-grids.parquet`: Grids emparejados que definen las unidades tratadas y de control

**Outputs generados:**
- Resultados de regresión copiados al portapapeles (coeficientes, errores estándar, valores p)

In [1]:
import os
import pandas as pd
import statsmodels.formula.api as smf


PATH_DATA = '../../data/'
INICIO_OPERACIONES = pd.to_datetime("2019-04-22")

In [2]:
outcome = pd.read_parquet(os.path.join(PATH_DATA, "outcome-variables.parquet"))
matched_grids = pd.read_parquet(os.path.join(PATH_DATA, "matched-grids.parquet"))

In [3]:
def get_panel_data(variable:str) -> pd.DataFrame:
    """
    Contstruir el panel data utilizando como outcome la variable especificada.
    
    Args: 
        - variable: {total, min, pic, fcs}
        
    Return: 
        - DataFrame with columns outcome_total, outcome_tasas, treat, post, treat_post, grid_id, timestamp
    """
    panel_data = (
        outcome
        .merge(matched_grids[["grid_id", "has_camera"]], on='grid_id')
        .rename({
            "has_camera":"treat"
        }, axis=1)
        .assign(
            outcome_total=lambda x: x[variable],
            outcome_tasas=lambda x: x[variable] / x.volumen_mensual,
            post=lambda x: (x.timestamp >= INICIO_OPERACIONES).astype(int),
            treat_post=lambda x: x.treat * x.post
        )
        [["outcome_total", "outcome_tasas", "treat", "post", "treat_post", "grid_id", "timestamp"]]
    )
    return panel_data

In [4]:
response = {}

outcome_variables = ["total", "tasas"]
for outcome_variable in outcome_variables:
    response[outcome_variable] = {}

    variables = ["total", "min", "pic", "fcs"]
    for variable in variables:
        response[outcome_variable][variable] = {}

        # Generamos los datos de panel
        panel_data = get_panel_data(variable)

        # Modelos
        model = smf.ols(
            formula=f"outcome_{outcome_variable} ~ treat + post + treat_post",
            data=panel_data
        ).fit(cov_type="HC1")

        fe_model = smf.ols(
            formula=f"outcome_{outcome_variable} ~ treat_post + C(grid_id) + C(timestamp)",
            data=panel_data
        ).fit(cov_type="cluster", cov_kwds={"groups": panel_data["grid_id"]})

        # Coeficientes a capturar
        coef_order = ["Intercept", "treat", "post", "treat_post"]

        result_list = []

        for coef_name in coef_order:
            # Coef y SE del modelo sin FE
            if coef_name in model.params.index:
                coef = round(model.params[coef_name], 10)
                se = round(model.bse[coef_name], 10)
                pval = round(model.pvalues[coef_name], 10)
            else:
                coef, se, pval = (float("nan"), float("nan"), float("nan"))

            # Solo treat_post tiene estimación en FE (los otros están absorbidos)
            if coef_name == "treat_post":
                pval_fe = fe_model.pvalues.get("treat_post", "")
                if pval_fe != "": pval_fe = round(pval_fe, 10)
            else:
                pval_fe = ""

            result_list.append({
                "Coeficiente (err. estándar)": f"{coef:.4g} ({se:.4g})" if not pd.isna(coef) else "nan",
                "Valor p": pval,
                "Valor p con efectos fijos": pval_fe
            })

        response[outcome_variable][variable] = result_list

In [15]:
pd.DataFrame(response.get('tasas').get('min')).T.set_index(0).T.to_clipboard(index=False)